In [33]:
import os
import json
import subprocess
import wget
from tqdm import tqdm  # 用於顯示進度條

# 定義下載資料夾路徑
folderPath = "./youtube_downloads"

# 確保下載資料夾存在


def ensure_folder_exists():
    if not os.path.exists(folderPath):
        os.makedirs(folderPath)
        print(f"已創建資料夾: {folderPath}")

# 下載 yt-dlp 執行檔


def download_ytdlp():
    ytdlp_path = './yt-dlp.exe'
    if not os.path.exists(ytdlp_path):
        print('[下載 yt-dlp]')
        try:
            wget.download(
                'https://github.com/yt-dlp/yt-dlp/releases/latest/download/yt-dlp.exe', ytdlp_path)
            print("\nyt-dlp 下載完成")
        except Exception as e:
            print(f"下載 yt-dlp 失敗: {e}")
            return False
    return True

# 獲取播放清單信息並創建 JSON 檔案


def create_playlist_json(playlist_url, limit=None):
    print(f"正在獲取播放清單信息: {playlist_url}")

    # 使用 yt-dlp 獲取播放清單信息
    cmd = [
        './yt-dlp.exe',
        playlist_url,
        '--flat-playlist',
        '--dump-json'
    ]

    # 如果有設定限制數量
    if limit:
        cmd.extend(['--playlist-items', f'1-{limit}'])

    try:
        process = subprocess.Popen(
            cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
        stdout, stderr = process.communicate()

        if process.returncode != 0:
            print(f"獲取播放清單信息失敗")
            try:
                print(f'錯誤信息: {stderr.decode("utf-8")}')
            except UnicodeDecodeError:
                print(f'錯誤信息: {stderr.decode("cp950", errors="replace")}')
            return None

        # 解析每行 JSON
        videos = []
        for line in stdout.decode('utf-8', errors='replace').strip().split('\n'):
            if line:
                try:
                    video_info = json.loads(line)
                    videos.append({
                        "link": f"https://www.youtube.com/watch?v={video_info['id']}",
                        "title": video_info.get('title', 'Unknown Title')
                    })
                except json.JSONDecodeError:
                    continue

        # 寫入 JSON 檔案
        json_path = f"{folderPath}/youtube.json"
        with open(json_path, 'w', encoding='utf-8') as f:
            json.dump(videos, f, ensure_ascii=False, indent=2)

        print(f"已創建 JSON 檔案，包含 {len(videos)} 個影片")
        return videos

    except Exception as e:
        print(f"處理播放清單時發生錯誤: {e}")
        return None

# 下載影片


def download_videos(video_list, limit=None, format='b[ext=mp4]'):
    if not video_list:
        print("沒有影片可下載")
        return

    # 限制下載數量
    if limit and limit < len(video_list):
        videos_to_download = video_list[:limit]
        print(f"將下載前 {limit} 個影片（共 {len(video_list)} 個）")
    else:
        videos_to_download = video_list
        print(f"將下載全部 {len(video_list)} 個影片")

    # 使用 tqdm 顯示整體進度
    for i, video in enumerate(tqdm(videos_to_download, desc="整體進度")):
        try:
            link = video.get('link')
            title = video.get('title', '未知標題')
            if not link:
                print(f"錯誤: 第 {i+1} 個項目缺少連結")
                continue

            print(f"\n正在下載 ({i+1}/{len(videos_to_download)}): {title}")

            # 定義輸出檔案名稱模板
            output_template = f'{folderPath}/%(title)s-%(id)s.%(ext)s'

            # 定義指令
            cmd = [
                './yt-dlp.exe',
                link,
                '-f', format,
                '-o', output_template,
                '--progress'  # 顯示下載進度
            ]

            # 執行指令並處理可能的編碼錯誤
            try:
                process = subprocess.Popen(
                    cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
                stdout, stderr = process.communicate()

                # 檢查結果
                if process.returncode == 0:
                    print(f'✓ 下載成功: {title}')
                else:
                    print(f'✗ 下載失敗: {title}')
                    try:
                        print(f'錯誤信息: {stderr.decode("utf-8")}')
                    except UnicodeDecodeError:
                        print(
                            f'錯誤信息: {stderr.decode("cp950", errors="replace")}')
            except Exception as e:
                print(f"執行下載命令時發生錯誤: {e}")

        except Exception as e:
            print(f"處理影片時發生錯誤: {e}")

# 主函數


def main():
    # 確保資料夾存在
    ensure_folder_exists()

    # 下載 yt-dlp
    if not download_ytdlp():
        return

    # 播放清單 URL
    playlist_url = input("請輸入 YouTube 播放清單 URL: ")
    if not playlist_url:
        print("未提供 URL，程序結束")
        return

    # 詢問是否限制數量
    limit_input = input("要限制下載數量嗎？(輸入數字，或按 Enter 不限制): ")
    limit = int(limit_input) if limit_input.isdigit() else None

    # 詢問下載格式
    format_input = input(
        "請選擇下載格式 (輸入數字):\n1. 最佳影片和音訊 (預設)\n2. 僅音訊 (MP3)\n3. 1080p\n4. 720p\n選擇: ")

    # 設定格式
    if format_input == '2':
        format = 'ba[ext=m4a]'
    elif format_input == '3':
        format = 'bestvideo[height<=1080]+bestaudio/best[height<=1080]'
    elif format_input == '4':
        format = 'bestvideo[height<=720]+bestaudio/best[height<=720]'
    else:
        format = 'b[ext=mp4]'  # 預設最佳格式

    # 創建 JSON 檔案並獲取影片列表
    video_list = create_playlist_json(playlist_url, limit)

    if video_list:
        print("\n已成功創建 youtube.json 檔案")

        # 詢問是否立即下載
        download_now = input("是否立即下載這些影片？(y/n, 預設: y): ").lower()
        if download_now != 'n':
            # 下載影片
            download_videos(video_list, limit, format)
            print("\n所有下載已完成")
        else:
            print("\nJSON 檔案已創建，但不進行下載")

    print("\n處理完成")


# 執行主函數
if __name__ == "__main__":
    main()

正在獲取播放清單信息: https://www.youtube.com/watch?v=G3UMvhSQ3LQ
已創建 JSON 檔案，包含 1 個影片

已成功創建 youtube.json 檔案
將下載全部 1 個影片


整體進度:   0%|          | 0/1 [00:00<?, ?it/s]


正在下載 (1/1): 【EP01純享版】脫學者｜品怡〈我是一隻魚〉 YOUNG VOICE Live Version


整體進度: 100%|██████████| 1/1 [00:13<00:00, 13.82s/it]

✓ 下載成功: 【EP01純享版】脫學者｜品怡〈我是一隻魚〉 YOUNG VOICE Live Version

所有下載已完成

處理完成
